In [10]:
pip install scikit-learn

     ---------------------------------------- 8.1/8.1 MB 9.2 MB/s eta 0:00:00
     ---------------------------------------- 36.6/36.6 MB 9.8 MB/s eta 0:00:00
     ------------------------------------- 309.1/309.1 kB 19.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import numpy as np

# Load the data
D = np.genfromtxt("data/lines.csv", delimiter=",", skip_header=1)

# Extract data for the first line (x1 and y1)
x1 = D[:, 0]
y1 = D[:, 3]

def total_least_squares(x, y):
    # Center the data
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    x_centered = x - x_mean
    y_centered = y - y_mean
    
    # Construct the matrix for SVD
    A = np.vstack([x_centered, y_centered]).T
    
    # Perform SVD
    _, _, Vh = np.linalg.svd(A)
    

    a, b = Vh[-1, :]
    
    # Solve for slope (m) and intercept (c) in y = mx + c
    # Since ax + by + d = 0 -> y = (-a/b)x - (d/b)
    slope = -a / b
    intercept = y_mean - slope * x_mean
    
    return slope, intercept

m1, c1 = total_least_squares(x1, y1)
print(f"First Line Parameters: Slope = {m1:.4f}, Intercept = {c1:.4f}")

First Line Parameters: Slope = 1.2207, Intercept = -5.9872


In [ ]:
from sklearn.linear_model import RANSACRegressor

# Prepare the flattened data
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten().reshape(-1, 1)
Y_all = Y_cols.flatten()

# Storage for results
remaining_X = X_all
remaining_Y = Y_all
lines = []

for i in range(3):
    # Initialize RANSAC
    ransac = RANSACRegressor(residual_threshold=0.5) # Adjust threshold based on noise
    ransac.fit(remaining_X, remaining_Y)
    
    # Identify the 'Inliers' (points that fit the line)
    inlier_mask = ransac.inlier_mask_
    outlier_mask = np.logical_not(inlier_mask)
    
    # Store the parameters
    m = ransac.estimator_.coef_[0]
    b = ransac.estimator_.intercept_
    lines.append((m, b))
    
    print(f"Line {i+1}: Slope = {m:.4f}, Intercept = {b:.4f}")
    
    # Remove the points accounted for
    remaining_X = remaining_X[outlier_mask]
    remaining_Y = remaining_Y[outlier_mask]

    if len(remaining_X) == 0:
        break

Line 1: Slope = -0.4496, Intercept = 2.1993
Line 2: Slope = 1.0287, Intercept = 1.0070
Line 3: Slope = 1.2734, Intercept = -6.0581
